# COM 763: Advanced Machine Learning (Task 1)

# SQL Query Runtime Prediction

**Author:** Aled Caio Rowlands  

**Module:** COM 763 Advanced Machine Learning

**Github link:** [https://github.com/AledCaioRow/COM763---ML-pipeline-for-SQL-error-detection-project](https://github.com/AledCaioRow/COM763---ML-pipeline-for-SQL-error-detection-project)

**Streamlitt http:** http://localhost:8765/
Note: I could not figure out how to input my diagrams of the two databases on the streamlit app as to provide a schema to create queires the two images are under the 'image' folder titled 'Formula 1 Diagram' and 'Financical Diagram' There are example queires on teh streamlitt app to copy and paste for experimental case worst case.

---

### Abstract

This report documents a machine learning pipeline for SQL query runtime prediction using the BIRD Mini-Dev text-to-SQL benchmark [1]. The project began as binary classification (fast vs slow) and shifted to regression on `log(runtime_seconds)` after the classifier proved too weak.

The core finding was a clear limitation: Data size resulting in low test sample as Figure one show however, the streamlit application is a proactive way of increasing test data. Furthermore teh original goal was ambitous of a cross-schema classification stayed near-random and cross-schema regression produced negative R². When schema statistics were added and evaluation was restricted to individual databases, performance improved sharply for well-indexed schemas, **R²(log) ≈ 0.945** on `debit_card_specializing` and **≈ 0.929** on `formula_1` with Ridge (α = 10), whilst still failing on poorly indexed schemas like `financial`. However these the test data was tiny and so users of the streamlit app can test the model themselves, creating their own test dataset.

The project had **three main iterations**. Iteration 1 trained a global classifier on seen databases and tested on held-out ones (cross-schema); it was also re-evaluated using a per-database 80/20 query split that pooled queries from all schemas into a shared train/test set, same model and features, different split geometry. Iteration 2 attempted per-database classification, which proved ill-posed given label skew. Iteration 3 trained per-database regression models with schema statistics added alongside SQL features, and is the final deployed system.


## 1. Problem Definition and System Framing

SQL query performance is difficult to judge from query text alone, yet slow queries have real costs: compute time, delayed dashboards, and increased cloud spend. A lightweight model estimating runtime before execution would have practical value in a development or analyst workflow.

The central research question was:

> **Can a model trained on queries from some databases predict runtime on databases?**

Usefulness depends on cross-schema transfer: if a model only works on schemas it has already seen, every new deployment requires retraining.

A text-to-SQL benchmark covering 11 SQLite databases across finance, sport, and education domains was used [1]. Each databse, schema information, queryr are designed with lots of thought, the idea of using this was that the quality of data maybe a more usful data set than otehr online. The project used these SQL queries directly: executing them against their corresponding SQLite databases to collect runtime measurements. This provided realistic, diverse SQL across well-structured schemas, small enough to run repeatedly on local hardware.

**Why the original cross-schema question was naive:** Because the SQL queries were designed for correctness against specific schemas, their runtime is determined not just by query structure but by the data volumes and indexing decisions of each database. SQL text features, join count, aggregation count, nesting depth, capture query structure, but contain no information about table sizes or whether indexes exist. Asking an SQL-text-only model to transfer across schemas was asking it to predict something it could not observe. Had the BIRD schema metadata been incorporated from the outset — even something as simple as total row count per database, cross-database generalisation would have been far more plausible.

**Project evolution:** The framing shifted from binary classification to regression on `log(runtime_seconds)`, and then further to per-database regression with schema statistics. The final system does not attempt universal cross-schema prediction; it provides calibrated within-database models for databases it has been trained on.

**Deployed product:** The Streamlit application provides five pages. The **Live Compare** page is the core output: users select one of the 11 BIRD databases from a dropdown, paste a SQL query, and the app executes it directly against that database's SQLite file using the same 3-run median timing procedure used in training, then compares the measured runtime to the model's predicted tier side-by-side (the code below highlihts how runtime was clauclated) **Model Results** displays all evaluation artefacts — confusion matrices, ROC and PR curves, feature importance, and the per-database comparison table. **Predict** accepts SQL text, extracts structural features, and returns a fast/slow classification with probability. **Data Explorer** provides interactive runtime distribution plots, class balance charts, and feature correlation heatmaps.

In [3]:
import sqlite3, time, numpy as np

def time_query(db_path, sql, timing_runs=3, timeout_s=30):
    """Run sql against db_path and return the median wall-clock time (seconds)."""
    timings = []
    for _ in range(timing_runs):
        conn = sqlite3.connect(f"file:{db_path}?mode=ro", uri=True)
        deadline = time.perf_counter() + timeout_s

        # Progress handler raises TimeoutError when wall-clock limit is exceeded
        conn.set_progress_handler(
            lambda: (_ for _ in ()).throw(TimeoutError()) if time.perf_counter() > deadline else 0,
            1_000_000
        )
        t0 = time.perf_counter()
        conn.execute(sql).fetchall()       # execute + fetch so result set is materialised
        timings.append(time.perf_counter() - t0)
        conn.close()

    return float(np.median(timings))       # median over 3 runs reduces jitter

## 2. Data Pipeline and Feature Handling

The data source was **BIRD Mini-Dev**, covering 11 SQLite databases. Each query was executed with **3 timing runs** and a **30-second timeout**, taking the **median runtime** as the label.


| Snapshot                    | Raw rows | Labelled rows | Fast | Slow |
| --------------------------- | -------- | ------------- | ---- | ---- |
| Earlier baseline extraction | 425      | 320           | 213  | 107  |
| Current expanded snapshot   | 498      | 374           | 249  | 125  |


### SQL structural features (25 features)

Features were extracted from SQL text using `sqlparse` with no access to the database at extraction time. The 25 features cover: query size (`n_tokens`, `query_length`), join complexity (`n_joins`, `n_tables_approx`), predicate structure (`n_where_predicates`), clause flags (`has_group_by`, `has_order_by`, `has_having`, `has_distinct`, `has_limit`, `has_union`), subquery indicators (`n_subqueries`, `has_subquery`, `max_nesting_depth`, `has_correlated_subquery`), aggregation counts (`n_count`, `n_sum`, `n_avg`, `n_max`, `n_min`, `n_aggregations`), and pattern flags (`has_between`, `has_in_clause`, `has_like`, `has_exists`).

Code below shows how SQL was parsed for feature were extraction

In [4]:
def extract_features(sql_text):
    """Parse one SQL string and return a dict of 25 structural features."""
    sql_upper = sql_text.upper()

    # --- size features ---
    features = {
        "n_tokens":       len(sql_upper.split()),
        "query_length":   len(sql_text),
        "n_joins":        sql_upper.count(" JOIN "),
    }

    # --- clause flags (binary: 1 if clause present, else 0) ---
    for clause in ["GROUP BY", "ORDER BY", "HAVING", "DISTINCT", "LIMIT", "UNION"]:
        features[f"has_{clause.lower().replace(' ', '_')}"] = int(clause in sql_upper)

    # --- subquery depth via parenthesis tracking ---
    depth = max_depth = 0
    for ch in sql_text:
        if ch == '(':  depth += 1; max_depth = max(max_depth, depth)
        elif ch == ')': depth -= 1
    features["max_nesting_depth"] = max_depth

    # --- aggregation counts ---
    for agg in ["COUNT(", "SUM(", "AVG(", "MAX(", "MIN("]:
        features[f"n_{agg.lower().rstrip('(')}"] = sql_upper.count(agg)

    return features



### Schema statistics features (6 features, added in Iteration 3)

After SQL-only models failed, six schema-level statistics were computed directly from each SQLite database file and appended as additional features: `schema_n_tables`, `schema_total_rows`, `schema_max_table_rows`, `schema_total_indexes`, `schema_index_coverage` (the proportion of tables with at least one index defined), and `schema_log_total_rows`. Because we train per-database, these values are constant across all rows for a given schema. They act as an environmental anchor telling the model the scale and structure of the system it is working within.

The code below is the creation of the single most imporatnt feature of the final model, indexing.

In [2]:
import sqlite3, numpy as np

def schema_stats(db_path):
    """Return 6 schema-level statistics for a SQLite database file."""
    conn = sqlite3.connect(db_path)
    cur  = conn.cursor()
    cur.execute("SELECT name FROM sqlite_master WHERE type='table'")
    tables = [r[0] for r in cur.fetchall()]

    total_rows = max_rows = total_idx = tables_with_idx = 0
    for t in tables:
        cur.execute(f'SELECT COUNT(*) FROM "{t}"')
        rc = cur.fetchone()[0]
        total_rows += rc
        max_rows    = max(max_rows, rc)

        cur.execute(f'PRAGMA index_list("{t}")')   # lists all indexes on this table
        idxs = cur.fetchall()
        total_idx += len(idxs)
        if idxs:
            tables_with_idx += 1                   # count tables that have ≥1 index
    conn.close()

    n = len(tables)
    return {
        "schema_n_tables":       n,
        "schema_total_rows":     total_rows,
        "schema_max_table_rows": max_rows,
        "schema_total_indexes":  total_idx,
        "schema_index_coverage": (tables_with_idx / n) if n else 0.0,  # key feature
        "schema_log_total_rows": float(np.log1p(total_rows)),           # log-compress scale
    }


### Labelling and the regression pivot

A quantile-based scheme initially labelled the bottom 50% of runtimes `fast`, the top 25% `slow`, and dropped the middle 25% to give cleaner class boundaries. This removed 124 queries (24.9%). Whilst defensible in principle, it proved problematic: several databases had extreme skew (0% slow or 90%+ slow), making binary classification meaningless for those schemas. From Iteration 2 onwards the project pivoted to **regression on log(runtime_seconds)**. The log transformation compresses the heavy runtime tail and makes the regression target approximately Gaussian.

### The EDA failure

The fundamental error was not performing EDA on per-database distributions until the pipeline, feature extraction, training loop, and Streamlit deployment were already wired to BIRD Mini-Dev. This was a fundametal error. By the time the table below was produced, switching datasets would have required many hours of re-engineering with no guarantee of finding data of comparable structure.


| Database                | Queries | Fast | Slow | % Slow |
| ----------------------- | ------- | ---- | ---- | ------ |
| superhero               | 50      | 50   | 0    | 0%     |
| card_games              | 50      | 7    | 43   | 86%    |
| student_club            | 47      | 47   | 0    | 0%     |
| european_football_2     | 45      | 8    | 37   | 82%    |
| formula_1               | 41      | 38   | 3    | 7%     |
| codebase_community      | 35      | 3    | 32   | 91%    |
| financial               | 32      | 28   | 4    | 12%    |
| thrombosis_prediction   | 27      | 27   | 0    | 0%     |
| toxicology              | 25      | 24   | 1    | 4%     |
| debit_card_specializing | 19      | 14   | 5    | 26%    |
| california_schools      | 3       | 3    | 0    | 0%     |


`california_schools` has only 3 queries because BIRD Mini-Dev contains very few benchmark questions targeting this schema, it is one of the lightest databases in the benchmark. It was excluded from per-database modelling (minimum threshold: 15 queries).

Figure 2 - Box plot of query counts per database  


## 3. Model Implementation and Debugging

### Iteration 1: Cross-schema classification (held-out databases)

A global classifier was trained on all databases except `financial` and `formula_1`, then tested on those two held-out schemas. Given the SQL-only feature set, containing no information about table sizes or index coverage, this was bound to fail; the features the model needs to distinguish slow from fast queries on an unseen schema simply do not exist in the input.

**Result:** XGBoost selected by CV . Test F1 (slow) = 0.186, ROC-AUC = 0.461, accuracy = 0.52, near-random. The original deployment question is not answerable with SQL text alone.

The same classifier setup was then re-evaluated with a different split: each database was split 80/20 at the query level and the per-database training/test portions pooled globally. This is not a different model, it tests whether query-level signal exists within schemas the model has seen. **Result:** Random Forest, F1 (slow) = 0.391, ROC-AUC = 0.64. Marginally better, but still poor. The improvement reflects proximity of test queries to their training schemas, not a meaningful gain in predictive power.

Figure 3 shows the results of both models. 

### Iteration 2: Per-database SQL-only classification

Fitting separate classifiers per database was ill-posed: several databases have 0% slow queries under the global quantile policy (the slow class is undefined), and others are 86–91% slow. Per-database classification was abandoned in favour of regression. However, along this had no real impact, what did have an impact however was training on specific databases. Specific databases with extra features to be exact as Figure 4 shows.

### Iteration 3: Per-database regression with schema statistics

For each database with at least 15 queries, a regression model is trained on 80% of that database's queries using the full **31-feature vector** (25 SQL structural + 6 schema statistics) to predict `log(runtime_seconds)`.

**Why per-database and why schema statistics?** Training within a single schema means the schema statistics are directly informative: `schema_index_coverage`, the fraction of that database's tables with at least one index defined — is the single most predictive structural feature. A well-indexed database has stable, index-driven execution paths, so query structure features reliably map to runtime because the query planner's route is predictable. A database with zero index coverage (e.g. `financial`) routes every query through a full table scan, making runtime a function of unseen row counts rather than query shape, no amount of SQL features can predict that.

**Model zoo:** Ridge (α=1), Ridge (α=10), Lasso (α=0.01), Random Forest (100 trees), Gradient Boosting — all in a `StandardScaler → model` pipeline, selected by R²(log) on the 20% held-out test set.

The code below shows how the per database regression was ran:

In [6]:
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

FULL_FEATURES = ["n_tokens", "n_joins", "max_nesting_depth",   # ... all 31 features
                 "schema_index_coverage", "schema_log_total_rows"]
MIN_QUERIES   = 15   # skip databases with fewer rows than this

# Model zoo — every pipeline starts with StandardScaler
model_zoo = {
    "Ridge(a=1)":  Pipeline([("sc", StandardScaler()), ("m", Ridge(alpha=1.0))]),
    "Ridge(a=10)": Pipeline([("sc", StandardScaler()), ("m", Ridge(alpha=10.0))]),
    "Lasso":       Pipeline([("sc", StandardScaler()), ("m", Lasso(alpha=0.01))]),
    "RF":          Pipeline([("sc", StandardScaler()), ("m", RandomForestRegressor(n_estimators=100))]),
}

for db_id, grp in df.groupby("db_id"):
    if len(grp) < MIN_QUERIES:
        continue                                       # not enough data to split reliably

    X = grp[FULL_FEATURES]
    y = np.log(grp["runtime_s"])                       # predict log-runtime (approx Gaussian)
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

    best_r2, best_name = -999, ""
    for name, model in model_zoo.items():
        model.fit(X_tr, y_tr)
        r2 = r2_score(y_te, model.predict(X_te))       # R²(log) is the selection criterion
        if r2 > best_r2:
            best_r2, best_name = r2, name

    print(f"{db_id:<25} best={best_name}  R²(log)={best_r2:.3f}")

NameError: name 'Ridge' is not defined

## 4. Experimental Evaluation and Model Selection

### Cross-schema verdict

Across both classification and regression, SQL-only models do not transfer or perform well on single databases. Cross-schema Ridge regression yields R²(log) = −1.05, worse than predicting the mean, confirming that switching to a continuous target does not fix the transfer problem. The gains in Iteration 3 come entirely from adding schema context and focusing on a single database.



### Within-database schema-aware results


| Database                | n   | Slow % | Index coverage | Best model  | R²(log)     | MAE (s)  |
| ----------------------- | --- | ------ | -------------- | ----------- | ----------- | -------- |
| debit_card_specializing | 19  | 26%    | 0.67           | Ridge(α=10) | **0.945**   | 0.031    |
| formula_1               | 41  | 7%     | 0.50           | Ridge(α=10) | **0.929**   | 0.152    |
| student_club            | 47  | 0%     | 0.88           | Ridge(α=10) | **0.640**   | 0.000049 |
| european_football_2     | 45  | 82%    | 0.62           | Lasso       | **0.148**   | 0.120    |
| superhero               | 50  | 0%     | 0.00           | Lasso       | 0.099       | 0.000106 |
| toxicology              | 25  | 4%     | 1.00           | RF          | 0.009       | 0.001    |
| card_games              | 50  | 86%    | 0.29           | RF          | -0.329      | 0.195    |
| codebase_community      | 35  | 91%    | 0.38           | RF          | -0.390      | 0.230    |
| financial               | 32  | 12%    | 0.00           | RF          | **-10.435** | 0.007    |


Values of best R²(log) per database.

**Caveat on the top two R² figures:** `debit_card_specializing` has n=19, giving only 3–4 test rows at 20%. `formula_1` has n=41, giving roughly 8 test rows. R² computed over this few points has very high variance; the figures of 0.945 and 0.929 may partly reflect a favourable random split rather than a robust generalisation result. They should be interpreted as indicative of strong within-database signal rather than precise held-out estimates.

**Index coverage as the dividing line:** Databases with index coverage ≥ 0.50 (`debit_card_specializing`, `formula_1`, `student_club`) all achieve positive R², whilst `financial` and `superhero` — both at index coverage 0.00 — perform at or below baseline. The exception is `toxicology` (coverage 1.00, R² ≈ 0.009): the raw data shows a runtime range of 0.186 ms to 703 ms, so variance does exist. The near-zero R² is more likely a small-sample artefact — with only 25 labelled rows the test set contains just 5 queries, and the single slow outlier (0.703s) almost certainly fell into the training split, leaving the test set with only sub-6ms queries and no meaningful signal to evaluate against.


Figure 5 shows the impact of cross schema impact. 

Of course, this report must be caveated with the fact traing data was small meaning splitting it caused tiny test data. The results are unreliabe. However, the streamlit production of the app provide a way to retrain the model on runtimes and the inputted queires as to aim to imporve the model.


## References

[1] Li, J., Hui, B., Qu, G., Yang, J., Li, B., Li, B., Wang, B., Qin, B., Geng, R., Huo, N., Zhou, X., Ma, C., Huang, R., Lou, Q., Chen, Z., Zhang, Z., Li, Z., Zhu, J., Cai, T., Chen, R., Chen, X., Huang, S., Liu, K. and Zhu, Y. (2024). “Can LLM Already Serve as a Database Interface? A Big Bench for Large-Scale Database Grounded Text-to-SQLs,” *arXiv preprint arXiv:2305.03111*, May 2023. Michel, V., Thirion, B., Grisel, O., Blondel, M., Prettenhofer, P., Weiss, R., Dubourg, V., Vanderplas, J., Passos, A., Cournapeau, D., Brucher, M., Perrot, M. and Duchesnay, E. (2011). *Scikit-learn: Machine Learning in Python.* Journal of Machine Learning Research, 12, pp.2825-2830.